**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/andrews-2024"], check=True)
    os.chdir("/content/encore/entries/andrews-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Andrews, Geng, Voitiuk et al. (2024) from dandiset 001132

**Paper.** Andrews JP, Geng J, Voitiuk K, et al. (2024). Multimodal evaluation of network activity
and optogenetic interventions in human hippocampal slices. *Nature Neuroscience* 27, 2487-2499.
[doi:10.1038/s41593-024-01782-5](https://doi.org/10.1038/s41593-024-01782-5).

**Dataset.** [DANDI:001132](https://dandiarchive.org/dandiset/001132/0.241130.1903), version
`0.241130.1903` ([doi:10.48324/dandi.001132/0.241130.1903](https://doi.org/10.48324/dandi.001132/0.241130.1903)),
streamed from the archive. The paper names the dandiset without a version. There are two published
versions, 0.241119.1801 and 0.241130.1903; the data check below shows that they hold the same 69
assets (same asset identifiers and sizes), so the most recent one is pinned.

**Claims.** `claims/andrews-2024.yaml`, written from the NIH author manuscript in PubMed Central
(PMC11881784) with its Methods, legends and supplement, and checked by a second reader.

**What the dandiset holds.** Each of the 69 files holds one HD-MEA recording as raw voltage only:
no sorted units, no light (GPIO) events, no stimulation log, no condition or medium labels and no
session description. Almost every claim rests on the paper's curated units and on which recording
and which trials make each condition, so they are blocked. The notebook surveys every recording,
recovers what it can of the light events from the voltage, tests the one claim about the cohort,
and runs the paper's network model.

| Claim | Status |
|---|---|
| table1-cohort | tested |
| units-per-recording | blocked: data_missing (no sorted units) |
| transduction-range, reporter-gcl-pyramidal | blocked: data_missing (histology) |
| patch-hckcr1-hyperpolarizing | blocked: data_missing (patch clamp) |
| simulation-dampening | tested: the published model is built and run with NEST 3.4 from conda-forge |
| fig3a-spontaneous-reduction, physio-bic-reduction-all-slices, physio-bic-amplitude-unchanged, table2-fr-decrease-physio-bic, table2-fr-decrease-0mg, zero-mg-reduction, ka-light-reduction, chr2-increase, closed-loop-bicuculline, intensity-threshold, s3-reduction-by-condition | blocked: data_missing (units; light events are recoverable in 3C, 4C and 7D, and in no file of 5C, 8D, 10F or 11G) |
| zero-mg-isolated-bursts, ka-rhythmic-bursts, ka-bursts-during-light, ka-theta-coherence, fig4-firing-propagation, fig4-theta-propagation | blocked: data_missing (units; conditions not labelled) |
| transduction-not-explaining-response | blocked: data_missing (depends on the two Table 2 claims, which are blocked; its transduction values are an input from Table 2) |
| waveform-clusters, waveform-gcl-association, opto-response-clusters, opto-waveform-association, opto-cluster3-profile | blocked: data_missing (units) |

A variant on multi-unit threshold crossings for slice 3C, whose light steps reach the first channel
group, is shown below; it decides no outcome.

In [ ]:
import resource
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
from scipy import stats
from statsmodels.stats.multitest import multipletests

import hdmea

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", hdmea.DANDISET, "version", hdmea.VERSION)

## Data check

The files and versions, then the metadata of every file. Each check of the metadata against the values is stated under its output.

In [ ]:
versions = requests.get(f"{hdmea.API}/dandisets/{hdmea.DANDISET}/versions/", timeout=60).json()["results"]
print("versions:", [(v["version"], v["status"], v["asset_count"], v["size"]) for v in versions])
published = [v["version"] for v in versions if v["version"] != "draft"]
listing = {v: {(a["asset_id"], a["path"], a["size"]) for a in hdmea.assets(v)} for v in published}
print("published versions hold the same assets:", len({frozenset(s) for s in listing.values()}) == 1)
meta = requests.get(f"{hdmea.API}/dandisets/{hdmea.DANDISET}/versions/{hdmea.VERSION}/", timeout=60).json()
summ = meta["assetsSummary"]
print("assetsSummary: files", summ["numberOfFiles"], "subjects", summ["numberOfSubjects"],
      "bytes", summ["numberOfBytes"], "variables", summ["variableMeasured"])

In [ ]:
ASSETS = hdmea.assets()
def load_meta(a):
    url = hdmea.s3_url(a["asset_id"])
    return dict(recording=hdmea.recording_name(a["path"]), path=a["path"], size_gb=a["size"] / 1e9, url=url,
                **hdmea.metadata(url))
with ThreadPoolExecutor(8) as ex:
    M = list(ex.map(load_meta, ASSETS))
info = pd.DataFrame([{k: v for k, v in m.items() if k not in ("x", "y", "electrode")} for m in M])
info["slice"] = info.recording.str.split("-").str[0]
info["patient"] = info.slice.str[-1]
info["duration_s"] = info.n_samples / info.rate
order = info.slice.str[:-1].astype(int) * 100 + info.recording.str.split("-").str[1].astype(int)
info = info.assign(o=order).sort_values("o").drop(columns="o").reset_index(drop=True)
print(info[["recording", "subject_id", "age", "sex", "species", "session_start", "n_channels", "duration_s",
            "size_gb"]].to_string())

In [ ]:
print("files:", len(info), " slices:", info.slice.nunique(), " patients:", info.patient.nunique())
print("recordings per slice:", info.groupby("slice", sort=False).size().to_dict())
print("subject_id equals the recording name in every file:", (info.subject_id == info.recording).all())
print("age and sex constant within each patient:",
      info.groupby("patient").apply(lambda g: g[["age", "sex"]].drop_duplicates().shape[0] == 1, include_groups=False).all())
print("channels constant within each slice:", info.groupby("slice").n_channels.nunique().eq(1).all())
print("dtype", info.dtype.unique(), " sampling rate (Hz)", info.rate.unique(), " conversion (V/count)", info.conversion.unique(),
      " offset", info.offset.unique(), " unit", info.unit.unique())
print("chunks", info.chunks.astype(str).unique(), " filters", info.filters.astype(str).unique())
print("duration (s): min %.1f, median %.1f, max %.1f, total %.0f" % (info.duration_s.min(), info.duration_s.median(),
      info.duration_s.max(), info.duration_s.sum()))
print("session_start values:", sorted(info.session_start.str[:10].unique()))
print("session_description non-empty:", (info.session_description != "").sum())
print("electrode locations:", sorted({l for m in M for l in m["location"]}), " device:", info.device.unique())
print("files with a units table:", info.has_units.sum(), " with intervals:", info.intervals.sum(),
      " with processing modules:", sum(len(p) > 0 for p in info.processing),
      " with stimulus series:", sum(len(s) > 0 for s in info.stimulus),
      " acquisition objects per file:", info.n_acquisition.unique())
print("objects outside the ElectricalSeries and general/:", sorted({p for ps in info.paths for p in ps}))
xs = np.concatenate([m["x"] for m in M]); ys = np.concatenate([m["y"] for m in M]); el = np.concatenate([m["electrode"] for m in M])
print("electrode x (um): %.1f to %.1f; y: %.1f to %.1f; all on a 17.5 um grid: %s; electrode ids %d to %d" % (
      xs.min(), xs.max(), ys.min(), ys.max(), bool(np.allclose(np.r_[xs, ys] / 17.5, np.round(np.r_[xs, ys] / 17.5))),
      el.min(), el.max()))

In [ ]:
# Every field under general/ in every file (where NWB keeps virus, pharmacology, protocol, notes,
# optogenetics, experiment_description and the like), and the ElectricalSeries' own description and comments
gen = pd.DataFrame([{**m["general"], **{f"ElectricalSeries {k}": v for k, v in m["series_text"].items()}} for m in M],
                   index=[m["recording"] for m in M])
print("fields under general/ in any file:")
for col in gen.columns:
    vals = gen[col].dropna().astype(str).unique()
    shown = (": " + " | ".join(v.replace("\n", " ") for v in vals)) if len(vals) <= 2 else ""
    print(f"  {col}: in {gen[col].notna().sum()} files, {len(vals)} distinct value(s){shown}")
nwb_fields = ("virus", "pharmacology", "protocol", "notes", "optogenetics", "slices", "surgery", "stimulus",
              "data_collection", "subject/genotype", "subject/strain", "subject/description")
print("NWB fields present in any file:", {k: any(c == k or c.startswith(k + "/") for c in gen.columns) for k in nwb_fields})
words = ("hckcr1", "chr2", "kcr", "opsin", "virus", "aav", "camk", "eyfp", "bicuc", "kainic", "magnesium", "0-mg",
         "0mg", "led", "light", "stim", "pharmacolog", "medium", "media", "baseline", "closed")
hits = sorted({(col, w) for col in gen.columns for w in words
               if any(w in (col + " " + v).lower() for v in gen[col].dropna().astype(str))})
print("opsin, virus, drug, medium or light words in any of these fields or their names:", hits)

The files are named by slice (`1A` to `12G`, the slice IDs of Table 1) and a recording index; the
letter is the patient. Each file's `subject_id` is the recording (`3C-0`), so the archive counts
69 subjects where there are 7 patients and 12 slices. Every `session_start_time` is 2024-09-28, the
conversion date, not the recording date (the paper's code names recordings from 2023). The
sampling rate (20 kHz), the channel counts (up to 1,024) and the electrode positions on the
17.5 µm grid of a 26,400-electrode MaxWell array are consistent with the Methods. Nothing in the
files says which medium, drug or light protocol a recording belongs to, which opsin a slice
carries, or which of the paper's recordings a file is. Under `general/` every file holds only the
devices, the electrode table and group, `experiment_description` (the paper's title, the same in
all 69), `institution`, `lab`, `source_script` and the subject's age, sex, species and
`subject_id`; there is no `virus`, `pharmacology`, `protocol`, `notes`, `optogenetics`,
`stimulus`, `slices` or `surgery` field, no subject genotype, strain or description, and no
opsin, virus, drug, medium or light word in any field. The ElectricalSeries' description and
comments are the converter's defaults.

## Survey of every recording

One pass over the first 64-channel group of every file gives the channel-mean voltage in 10 ms
bins (to find light steps) and multi-unit threshold crossings per channel in 100 ms bins (helper
`hdmea.scan`; parameters and their sources are in its docstring). It reads about 13 GB and takes
about 12 minutes. It was timed on two recordings while preparing (11 s for a 190 s recording,
1.3 GB peak memory).

In [ ]:
t = time.time()
SCAN = {}
for m in M:
    SCAN[m["recording"]] = hdmea.scan(m["url"])
print("scanned %d recordings in %.1f min; %.1f GB compressed read; peak memory %.2f GB" % (
      len(SCAN), (time.time() - t) / 60, sum(s["compressed"] for s in SCAN.values()) / 1e9,
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
rng = pd.DataFrame([{"slice": r.split("-")[0], "vmin": s["vmin"], "vmax": s["vmax"],
                     "rail_frac_max": float(s["rail"].max()), "rail_frac_median": float(np.median(s["rail"]))}
                    for r, s in SCAN.items()])
rng = rng.groupby("slice", sort=False).agg(vmin=("vmin", "min"), vmax=("vmax", "max"),
                                           rail_frac_max=("rail_frac_max", "max"), rail_frac_median=("rail_frac_median", "max"))
rng["conversion_uV"] = info.groupby("slice").conversion.agg(lambda c: sorted(set(np.round(c * 1e6, 3)))).reindex(rng.index)
print("values (counts) of the first channel group per slice, the largest fraction of its channels at 0 or 1023 in any")
print("100 ms bin, and the conversion to volts:")
print(rng.round(3).to_string())

### Recordings deposited twice

The compressed bytes of the first and the last chunk of the first channel group, compared across files; files of equal length are listed too.

In [ ]:
dig = pd.DataFrame([{"recording": r, "first": s["digest_first"], "last": s["digest_last"], "n": s["n"]} for r, s in SCAN.items()])
print("files sharing a first chunk:", dig[dig.duplicated("first", keep=False)].recording.tolist())
print("files sharing a last chunk:", dig[dig.duplicated("last", keep=False)].recording.tolist())
same_len = dig[dig.duplicated("n", keep=False)].groupby("n").recording.apply(list)
print("files of identical length (samples):", same_len.tolist())

No two files share a chunk, so no recording is deposited twice; files of equal length differ in their data.

### Light events

The Methods say the light ON and OFF times were stored in the MaxWell file's GPIO bit and in a
stimulation log (Methods, Design of optogenetic system). Neither is in the NWB files. In slice 3C
the LED leaves a step in the voltage of every channel at onset and the opposite step at offset;
these steps are counted here at several thresholds against the bin-to-bin noise, between
consecutive 10 ms bins (lag 1) and between bins 20 ms apart (lag 2, which also finds a step split
across two bins).

In [ ]:
rows = []
for r, s in SCAN.items():
    row = {"recording": r, "duration_s": round(s["n"] / hdmea.FS, 1)}
    for th in (20, 40, 60, 100):
        L = hdmea.light_steps(s["dc"], th)
        row[f"trials@{th}"] = len(L["on"])
        row[f"up/down@{th}"] = f"{L['n_up']}/{L['n_down']}"
    for th in (40, 60):
        row[f"trials@{th}_lag2"] = len(hdmea.light_steps(s["dc"], th, lag=2)["on"])
    row["noise"] = round(L["noise"], 2)
    row["max_step"] = round(float(np.abs(np.diff(s["dc"])).max()), 1)
    row["max_step_lag2"] = round(float(np.abs(s["dc"][2:] - s["dc"][:-2]).max()), 1)
    rows.append(row)
steps = pd.DataFrame(rows)
print(steps.to_string())
LIGHT = {r: hdmea.light_steps(SCAN[r]["dc"]) for r in steps.recording}
with_light = [r for r, L in LIGHT.items() if len(L["on"])]
print("\nrecordings with light trials at the main threshold (%g counts):" % hdmea.STEP_MIN, with_light)
for r in with_light:
    L = LIGHT[r]
    print(r, "onsets (s):", np.round(L["on"], 2).tolist(), " ON durations (s):", np.round(L["off"] - L["on"], 2).tolist(),
          " onset steps:", np.round(L["up"]).astype(int).tolist(), " offset steps:", np.round(L["down"]).astype(int).tolist())
print("recordings with light trials at 60 counts, lag 2:", steps[steps["trials@60_lag2"] > 0].recording.tolist(),
      "; at 40 counts, lag 2:", steps[steps["trials@40_lag2"] > 0].recording.tolist())

Paired light steps (onset +67 to +168 counts, offset -67 to -88 counts, in 10 ms bins against a
bin-to-bin noise of 0.5 to 1.6 counts) occur in the eight 3C recordings. In the first channel
group no other file has one at the main threshold of 60 counts, at lag 1 or lag 2: outside 3C,
one file (2B-1, noise 2.5) has a single unpaired upward jump of 73 counts, and the largest jump in
any other file is 57 counts (10F-7), unpaired. At 20 and 40 counts the extra steps at lag 1 are
also unpaired; at 40 counts and lag 2, 2B-1 has one paired step (largest lag-2 change 139.6
counts, against a noise of 2.5). The first group alone does not settle the other slices; every
group is read in the next section. The 3C light steps are in the voltage itself, so they share
its clock by construction; there is no trial table to compare with.

Each light period lasts 10.0 s (3C-0: 5.0 s). 3C-6 and 3C-7 hold six light periods each, about 20 s
apart (onset to onset); in 3C-6 the first starts 1 s into the file, so 5 of its 6 have a full 10 s
pre window. 3C-1 to 3C-5 hold one 10 s period each, starting 10 s into a 40 to 70 s file, which fits
the intensity sweeps of Extended Data Fig. 7, but the files do not say so. The claims file has five
trials for 3C (Fig. 3a, Extended Data Fig. 4 legend); 3C-6 has five whose three windows lie
in the file. Its onsets are 18 to 21 s apart, so some of these windows hold part of a
neighbouring light period (printed in the variant below).

### Light steps on every channel group

The scan above reads the first 64-channel group only. A light artifact confined to another part
of the array would be missed there, so every group of every file of the slices the paper
illuminated outside 3C in 10-trial, ChR2 or closed-loop protocols (4C, 5C, 7D, 8D, 10F, 11G) is
read here for its channel-mean trace (helper `hdmea.group_means`), with 3C as a positive control.
Steps are counted at lag 1 and at lag 2 (a step split across two 10 ms bins), at 60 and 40
counts. This reads about 130 GB; one 1.9 GB file took 14 s while preparing (0.8 GB peak memory),
so the whole was estimated at 16 minutes.

In [ ]:
t = time.time()
ALLG = {m["recording"]: hdmea.group_means(m["url"], workers=12) for m in M
        if m["recording"].split("-")[0] in ("3C", "4C", "5C", "7D", "8D", "10F", "11G")}
print("read every channel group of %d recordings in %.1f min (%.0f GB); peak memory %.2f GB" % (
      len(ALLG), (time.time() - t) / 60, info.set_index("recording").size_gb.reindex(list(ALLG)).sum(),
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))
rows = []
for r, D in ALLG.items():
    row = {"recording": r, "groups": len(D)}
    for th, lag in ((60, 1), (60, 2), (40, 2)):
        n = [len(hdmea.light_steps(d, th, lag=lag)["on"]) for d in D]
        row[f"groups_with_trials@{th}_lag{lag}"] = sum(k > 0 for k in n)
        row[f"most_trials@{th}_lag{lag}"] = max(n)
    row["max_step"] = round(float(max(np.abs(np.diff(d)).max() for d in D)), 1)
    row["max_step_lag2"] = round(float(max(np.abs(d[2:] - d[:-2]).max() for d in D)), 1)
    row["noise_median"] = round(float(np.median([hdmea.light_steps(d)["noise"] for d in D])), 2)
    rows.append(row)
allg = pd.DataFrame(rows)
print(allg.to_string())
out3c = allg[~allg.recording.str.startswith("3C")]
print("\noutside 3C, recordings with a light trial in any group: at 60 counts, lag 1:",
      out3c[out3c["groups_with_trials@60_lag1"] > 0].recording.tolist(), "; lag 2:",
      out3c[out3c["groups_with_trials@60_lag2"] > 0].recording.tolist(), "; at 40 counts, lag 2:",
      out3c[out3c["groups_with_trials@40_lag2"] > 0].recording.tolist())
print("outside 3C, largest step in any group: lag 1 %.1f, lag 2 %.1f counts" % (out3c.max_step.max(), out3c.max_step_lag2.max()))
for r in out3c[out3c["groups_with_trials@40_lag2"] > 0].recording:
    for g, d in enumerate(ALLG[r]):
        L = hdmea.light_steps(d, 40, lag=2)
        if len(L["on"]):
            print(r, "group", g, "onsets (s):", np.round(L["on"], 2).tolist(), "ON durations (s):",
                  np.round(L["off"] - L["on"], 2).tolist(), "steps:", np.round(L["up"]).astype(int).tolist(),
                  np.round(L["down"]).astype(int).tolist())

Every channel group of every 3C recording shows the light steps (16 of 16 groups, at every
threshold and lag), so the control works. Outside 3C no group of any file has a paired light trial at 60 counts
(the largest step between consecutive bins is 83.8 counts, in 5C-5). At lag 2,
paired steps of 40 to 88 counts, 10.0 s apart (in 4C-1 group 5 two steps fall below the threshold and one onset pairs with an offset 30 s later), appear in groups 5 to 14 of six files: 4C-1
(six periods 20 s apart, from 70.1 s), 4C-3 (one, at 30.0 s), 7D-2 (six, 30 s apart, from
110.2 s), 7D-7 (one, at 20.0 s), and, at 40 counts only and in groups 12 and 13, 7D-0 (three)
and 7D-5 (four). Their length, 10 s, matches the 3C light periods, though their spacing (20 s in 4C-1, 30 s in
7D-2 and 7D-0, 40 s in 7D-5) is not always the 20 s of 3C-6 and 3C-7; they are read as light
events, and the files do not say so. 5C-5 has one pair 0.76 s long in two groups, which does not fit
a 10 s light period. No group of any 5C, 8D, 10F or 11G recording shows a 10 s step, including
10F, which the paper illuminated for 10 trials, and the closed-loop slices 5C and 8D. Light
events can therefore be recovered for 3C, 4C (ChR2) and 7D, and for no file of 5C, 8D, 10F or
11G; 9E and 12G were read on their first channel group only, where they show none.

## table1-cohort

> Recordings came from 12 hippocampal slices from 7 adult patients with refractory temporal lobe
> epilepsy (5 female, 2 male, aged 20 to 52), 8 slices transduced with HcKCR1, 2 with ChR2 and 2
> with no opsin.

Criterion: reproduced when the dandiset's subjects and sessions resolve to 12 slices from 7
patients with the opsins, ages and sexes of Table 1; partially reproduced when the HcKCR1 slices
of Table 2 (3C, 5C, 7D, 8D, 9E, 10F, 11G, 12G) are all present and identifiable but others are not.
Table 1 is read from the paper (`paper/`); its ages and sexes per patient are A 20 M, B 44 F,
C 40 F, D 46 F, E 46 F, F 52 F, G 35 M.

In [ ]:
pat = info.groupby("patient").agg(age=("age", "first"), sex=("sex", "first"), slices=("slice", lambda s: sorted(set(s))),
                                  recordings=("recording", "size"))
pat["age_years"] = pat.age.str.extract(r"P(\d+)Y").astype(int)
print(pat.to_string())
print("\nslices:", info.slice.nunique(), " patients:", info.patient.nunique())
print("female patients:", (pat.sex == "F").sum(), " male:", (pat.sex == "M").sum(),
      " ages %d to %d" % (pat.age_years.min(), pat.age_years.max()))
# Table 1 of the paper (paper/): the slices, and each patient's age and sex
TABLE1_SLICES = ["1A", "2B", "3C", "4C", "5C", "6C", "7D", "8D", "9E", "10F", "11G", "12G"]
TABLE1_PATIENTS = {"A": (20, "M"), "B": (44, "F"), "C": (40, "F"), "D": (46, "F"), "E": (46, "F"), "F": (52, "F"), "G": (35, "M")}
print("slices as in Table 1:", sorted(set(info.slice)) == sorted(TABLE1_SLICES))
print("patients as in Table 1:", sorted(pat.index) == sorted(TABLE1_PATIENTS))
print("each patient's age and sex as in Table 1:",
      {p: (int(pat.loc[p, "age_years"]), pat.loc[p, "sex"]) == v for p, v in TABLE1_PATIENTS.items()})
hckcr1 = ["3C", "5C", "7D", "8D", "9E", "10F", "11G", "12G"]
print("HcKCR1 slices of Table 2 present by name:", all(s in set(info.slice) for s in hckcr1))
opsin_words = ("hckcr1", "chr2", "opsin", "virus", "aav", "camk", "eyfp", "genotype", "strain")
in_files = sorted({k for m in M for k, v in {**m["general"], **m["series_text"]}.items()
                   if any(w in (k + " " + str(v)).lower() for w in opsin_words)}
                  | {p for ps in info.paths for p in ps if any(w in p.lower() for w in opsin_words)})
print("fields naming an opsin, virus or genotype anywhere in the files (general/ included):", in_files,
      "; in the dandiset metadata:", any(w in str(meta).lower() for w in ("hckcr1", "chr2", "aav9")))

**Result: partially reproduced.** The 69 files resolve to 12 slices named as in Table 1 (1A to
12G), from 7 patients (A to G), 5 female and 2 male, aged 20 to 52, with each patient's age and sex
as in Table 1; the 8 HcKCR1 slices of Table 2 are present by name. The opsin of each slice is not
recorded in the files (no field under `general/` or elsewhere names an opsin, virus or genotype)
or in the dandiset's metadata, so the count of 8 HcKCR1, 2 ChR2 and 2 no-opsin slices can be had
only by reading the slice names against the paper's Table 1, not from the dandiset.

## Claims that need sorted units or light events

`units-per-recording` and every claim on firing rates, amplitudes, bursts, propagation, waveform
or opto-response clusters rests on the paper's curated units (Kilosort2, then manual and automatic
curation). The files hold no units table, so these claims are blocked as `data_missing`. The claims file makes a
re-sort a variant that decides nothing, and the manual curation cannot be repeated. The light-trial claims also need the light events, which are
recoverable from the voltage in slices 3C, 4C and 7D and in no file of 5C, 8D, 10F or 11G (above), and the claims on 0-mg, 0-mg+KA
and bicuculline recordings need to know which file is which condition, which no file says.

### Variant: the 3C light trials in the raw voltage

This is not the paper's method and decides no outcome. It shows what the dataset holds around the
3C light trials, the ones whose steps reach the first channel group that the scan reads. Per 100 ms bin of the first 64-channel group, it gives the
fraction of channels at 0 or 1023 (the ends of the range the values take) and the multi-unit
threshold crossings (300 to 6000 Hz, 5 MAD per channel and chunk). Crossings are counted in the
claims file's windows, pre (10 s before onset), ON (10 s from onset) and OFF (onset + 10 to
+ 20 s), averaged over the recording's trials, and compared by two-tailed paired t-tests over
channels with crossings, Holm-adjusted over the three comparisons, as the claims file's opening
comment does for units. Light onset and offset leave transients in these windows, so the
comparison is also run with the first second of each window left out (`skip_s` 1); the crossing
threshold is still estimated per 3.9 s chunk, transients included. A channel is not a unit: one channel can carry several units and one
unit can reach several channels.

In [ ]:
def compare(rec, rates):
    keep = (rates["pre"] + rates["on"] + rates["off"]) > 0
    pre, on, off = rates["pre"][keep], rates["on"][keep], rates["off"][keep]
    p = [stats.ttest_rel(a, b).pvalue for a, b in ((pre, on), (on, off), (pre, off))]
    padj = multipletests(p, method="holm")[1]
    return {"recording": rec, "trials": rates["n_trials"], "channels": int(keep.sum()),
            "pre_Hz": pre.mean(), "on_Hz": on.mean(), "off_Hz": off.mean(),
            "change_on_vs_pre_%": 100 * (on.mean() - pre.mean()) / pre.mean(),
            "p_pre_on": padj[0], "p_on_off": padj[1], "p_pre_off": padj[2],
            "channels_lower_on": int((on < pre).sum())}

def saturation(rec, skip):
    rail, on, ns = SCAN[rec]["rail"], LIGHT[rec]["on"], int(round(skip * 10))
    out = {"pre": [], "on": [], "off": [], "on_first_0.5s": []}
    for t in on:
        b = int(round(t * 10))
        if b - 100 < 0 or b + 200 > len(rail):
            continue
        out["pre"].append(rail[b - 100 + ns:b].mean()); out["on"].append(rail[b + ns:b + 100].mean())
        out["off"].append(rail[b + 100 + ns:b + 200].mean()); out["on_first_0.5s"].append(rail[b + ns:b + ns + 5].mean())
    return {f"rail_{k}": (np.mean(v) if v else np.nan) for k, v in out.items()}

# skip_s 0: the claims file's windows; skip_s 1: the first second of each window left out, which
# removes the onset and offset transients (the crossing threshold is still estimated per chunk)
res = []
for skip in (0.0, 1.0):
    for r in with_light:
        w = hdmea.window_rates(SCAN[r]["counts"], LIGHT[r]["on"], skip=skip)
        row = compare(r, w) if w["n_trials"] else {"recording": r, "trials": 0}
        res.append({"skip_s": skip, **row, **saturation(r, skip)})
var = pd.DataFrame(res)
print(var.round(4).to_string())

# window_rates keeps a trial when its three windows lie in the file, but does not check them against the
# neighbouring light periods; print, per trial of 3C-6 and 3C-7, the seconds of light each window holds
def light_in(a, b, periods):
    return sum(max(0.0, min(b, off) - max(a, on)) for on, off in periods)
for r in ("3C-6", "3C-7"):
    L = LIGHT[r]
    for k, t in enumerate(L["on"]):
        if t - 10 < 0 or (int(round(t * 10)) + 200) > len(SCAN[r]["counts"]):
            continue
        others = [(a, b) for j, (a, b) in enumerate(zip(L["on"], L["off"])) if j != k]
        print(r, "trial at %.2f s:" % t, ", ".join("%s %s %.2f s" % (w, s, light_in(t + a, t + b, others))
              for s in (0.0, 1.0) for w, (a, b) in (("pre", (-10 + s, 0)), ("ON", (s, 10)), ("OFF", (10 + s, 20)))
              ), "of light from other periods (skip_s 0, then 1)")


In [ ]:
# Crossings per second (summed over the 64 channels) and the fraction of channels at the rails,
# through the six light periods of 3C-7. A second is ON when it lies wholly inside a light period,
# "transition" when it holds an onset or an offset, and pre/off otherwise.
r = "3C-7"
c = SCAN[r]["counts"].sum(1)
per_s = c[: len(c) // 10 * 10].reshape(-1, 10).sum(1)
rail_s = SCAN[r]["rail"][: len(SCAN[r]["rail"]) // 10 * 10].reshape(-1, 10).max(1)
L = LIGHT[r]
state = np.full(len(per_s), "pre/off", dtype=object)
for a, b in zip(L["on"], L["off"]):
    state[int(np.ceil(a)):int(np.floor(b))] = "ON"
    state[int(np.floor(a))] = "transition"
    if int(np.floor(b)) < len(state):
        state[int(np.floor(b))] = "transition"
tab = pd.DataFrame({"crossings_per_s": per_s[: len(rail_s)], "max_rail_frac": rail_s, "state": state[: len(rail_s)]})
print(tab.groupby("state").agg(seconds=("crossings_per_s", "size"), median=("crossings_per_s", "median"),
                               min=("crossings_per_s", "min"), max=("crossings_per_s", "max"),
                               seconds_over_1000=("crossings_per_s", lambda v: int((v > 1000).sum())),
                               max_rail=("max_rail_frac", "max")).to_string())
first = int(np.floor(L["on"][0]))
print("seconds %d to %d around the first onset (%.2f s):" % (first - 3, first + 25, L["on"][0]))
print(tab.iloc[first - 3:first + 25].T.to_string())

**Result: blocked (data_missing); variant only.** fig3a-spontaneous-reduction needs the curated
units of 3C, which are not in the dandiset. In the raw voltage, light onset drives 81 to 94% of the
channels to the ends of their range in the first 0.5 s (4 of 64 channels, 6%, stay there through
the light in 3C-7), and seconds with over 1,000 crossings occur in 22 of the 54 seconds wholly
inside light in 3C-7, in 6 of the 12 seconds that hold an onset or offset, and in 8 of the other
124. With the claims file's windows, crossings fall by 86 to 90% from pre to ON in the five
single-trial files 3C-1 to 3C-5 (Holm-adjusted p 0.0001 or less), and not in the multi-trial
files 3C-6 (5 trials, -16%, adjusted p 0.82) and 3C-7 (6 trials, +6%, adjusted p 0.87), whose
onsets are 18 to 21 s apart, so that some pre and OFF windows hold part of a neighbouring light
period (printed above). With the first second of each window left out, 26 to 52%
of the channels are still at the ends of their range in the next 0.5 s; crossings fall by 89 to
93% in 3C-1 to 3C-5 (adjusted p 0.0001 or less), by 18% in 3C-6 (adjusted p 0.77), and rise by
17% in 3C-7 (adjusted p 0.06). Saturation and these transients confound the crossings, so the
variant cannot say how light changes firing.

## simulation-dampening

The claim is tested by running `Simulation/simulation.py` as published, at commit 6865bf6 of the
paper's repository. Its environment file and the versions on PyPI are printed below. The first
attempt runs the script's first step, NESTML's generation and build of the neuron model, in a
throwaway environment with the NEST that PyPI has and the NESTML the environment file pins. The
second, asked for by the round-2 verifier, takes NEST 3.4 from conda-forge, as the environment
file does, in a throwaway environment made with micromamba, and runs the whole script.


In [ ]:
import os, shutil, subprocess, tempfile
REPO = "https://raw.githubusercontent.com/braingeneers/AndrewsGengVoitiukEtAl_SourceCode/6865bf68cc3c8ea36a9b4f853b9248288fa1e52b"
env_yml = requests.get(f"{REPO}/Simulation/environment.yml", timeout=60)
env_yml.raise_for_status()
print("Simulation/environment.yml at 6865bf6:\n" + env_yml.text)
for pkg in ("nest-simulator", "nestml"):
    rel = requests.get(f"https://pypi.org/pypi/{pkg}/json", timeout=60).json()["releases"]
    print(f"{pkg} releases on PyPI:", sorted(rel))
model = requests.get(f"{REPO}/Simulation/models/izh_cond_exp2syn.nestml", timeout=60)
model.raise_for_status()
uv = shutil.which("uv")
print("uv:", uv)
with tempfile.TemporaryDirectory() as d:
    os.makedirs(f"{d}/models")
    open(f"{d}/models/izh_cond_exp2syn.nestml", "w").write(model.text)
    py = f"{d}/venv/bin/python"
    # the kernel's matplotlib backend is not one the throwaway environment knows, and NEST imports matplotlib
    env = {**{k: v for k, v in os.environ.items() if k != "MPLBACKEND"}, "PYNEST_QUIET": "1"}
    for cmd in ([uv, "venv", "-q", "--python", "3.10", f"{d}/venv"],
                [uv, "pip", "install", "-q", "--python", py, "nest-simulator==3.10.0", "nestml==5.*"]):
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        print(" ".join(cmd[1:3] + cmd[-2:]), "-> exit", out.returncode, out.stderr[-500:])
    ver = subprocess.run([py, "-c", "import nest, pynestml; print(nest.__version__, pynestml.__version__)"],
                         capture_output=True, text=True, env=env)
    print("import nest, pynestml -> exit", ver.returncode, "; versions:", ver.stdout.strip()[-40:], ver.stderr[-300:] if ver.returncode else "")
    gen = subprocess.run([py, "-c", "from pynestml.frontend.pynestml_frontend import generate_nest_target; "
                          f"generate_nest_target('{d}/models/', '{d}/build', module_name='hippocampusmodule', logging_level='WARNING')"],
                         capture_output=True, text=True, timeout=900, env=env)
    print("generate_nest_target exit", gen.returncode)
    print("\n".join(gen.stderr.strip().splitlines()[-3:]))

In [ ]:
# NEST 3.4 from conda-forge, as Simulation/environment.yml pins it, in a throwaway environment made with
# micromamba (2.10.0, from conda-forge), with the packages the file lists. The model is then built and
# Simulation/simulation.py run as published; each step's exit code and error are printed.
import io, json, tarfile
MM = "https://conda.anaconda.org/conda-forge/linux-64/micromamba-2.10.0-0.tar.bz2"
SIMDIR = tempfile.mkdtemp()
E = f"{SIMDIR}/env"
mm_tar = requests.get(MM, timeout=300)
mm_tar.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(mm_tar.content), mode="r:bz2") as t:
    t.extract("bin/micromamba", SIMDIR, filter="data")
os.makedirs(f"{SIMDIR}/run/models")
for f in ("simulation.py", "models/izh_cond_exp2syn.nestml"):
    src = requests.get(f"{REPO}/Simulation/{f}", timeout=60)
    src.raise_for_status()
    open(f"{SIMDIR}/run/{f}", "w").write(src.text)
simenv = {**{k: v for k, v in os.environ.items() if k != "MPLBACKEND"}, "MPLBACKEND": "Agg", "PYNEST_QUIET": "1",
          "PATH": f"{E}/bin:" + os.environ["PATH"]}

def step(label, cmd, **kw):
    out = subprocess.run(cmd, capture_output=True, text=True, env=simenv, **kw)
    print(f"{label} -> exit {out.returncode}")
    if out.returncode:
        print("   ", "\n    ".join([l for l in out.stderr.strip().splitlines() if "WARNING" not in l][-3:]))
    return out

GEN = ("from pynestml.frontend.pynestml_frontend import generate_nest_target; "
       f"generate_nest_target('models/', '{SIMDIR}/build', module_name='hippocampusmodule', logging_level='WARNING')")
# cmake 3 and a compiler are what NESTML 5 needs to build the module; the file does not list them
step("micromamba create python=3.10 nest-simulator=3.4 tqdm pip cmake=3 cxx-compiler make",
     [f"{SIMDIR}/bin/micromamba", "create", "-y", "-q", "-r", f"{SIMDIR}/mmroot", "-p", E, "-c", "conda-forge",
      "--override-channels", "python=3.10", "nest-simulator=3.4", "tqdm", "pip", "cmake=3", "cxx-compiler", "make"], timeout=1200)
step("pip install nestml==5.3.0 braingeneers==0.3.1.16 matplotlib",
     [f"{E}/bin/python", "-m", "pip", "install", "-q", "nestml==5.3.0", "braingeneers==0.3.1.16", "matplotlib"], timeout=1200)
# 1. as the environment file resolves today: odetoolbox is not pinned, and its newest release is taken
step("generate_nest_target, odetoolbox as resolved", [f"{E}/bin/python", "-c", GEN], cwd=f"{SIMDIR}/run", timeout=900)
# 2. odetoolbox 2.5, the release current when NESTML 5.2 and 5.3 came out (October 2022)
step("pip install odetoolbox==2.5", [f"{E}/bin/python", "-m", "pip", "install", "-q", "odetoolbox==2.5"], timeout=600)
shutil.rmtree(f"{SIMDIR}/build", ignore_errors=True)
step("generate_nest_target, odetoolbox 2.5", [f"{E}/bin/python", "-c", GEN], cwd=f"{SIMDIR}/run", timeout=900)
# 3. the conda-forge nest-config prints an empty --compiler, so the generated CMakeLists sets no C++
#    compiler; make it print the environment's compiler (the model is not changed)
cfg = open(f"{E}/bin/nest-config").read()
print("nest-config --compiler was:", repr(cfg.split("--compiler)")[1].split(";;")[0].strip()))
open(f"{E}/bin/nest-config", "w").write(cfg.replace('--compiler)\n        echo ""', f'--compiler)\n        echo "{E}/bin/x86_64-conda-linux-gnu-c++"', 1))
shutil.rmtree(f"{SIMDIR}/build", ignore_errors=True)
step("generate_nest_target, odetoolbox 2.5, nest-config compiler set", [f"{E}/bin/python", "-c", GEN], cwd=f"{SIMDIR}/run", timeout=900)
ver = step("versions", [f"{E}/bin/python", "-c", "import nest; from importlib.metadata import version as v; "
            "print('NEST', nest.__version__, '| ' + ' | '.join(p + ' ' + v(p) for p in ('nestml', 'odetoolbox', 'braingeneers', 'numpy')))"])
print(ver.stdout.strip())


In [ ]:
# Run Simulation/simulation.py unchanged (it builds the model again, then simulates opsin fractions 0, 0.10,
# 0.25 and 0.50 with its seed, 20) and keep each run's spike counts per 1 ms. As a check of the seed, the
# script's sim() is then run again with the same arguments and seeds 21 to 24 (a choice; the script has one).
DRIVER = '''
import json, runpy
import numpy as np
g = runpy.run_path("simulation.py", run_name="__main__")
def keep(sd, seed):
    return dict(seed=seed, p_opto=sd.metadata["p_opto"], n_cells=sd.N, length_ms=sd.length,
                threshold=sd.metadata["opto_threshold"], opto_times=[int(t) for t in sd.metadata["opto_times"]],
                counts=np.asarray(sd.binned(1)).astype(int).tolist())
runs = [keep(sd, 20) for sd in g["sds_fraction"]]
for seed in (21, 22, 23, 24):
    for p in (0.0, 0.1, 0.25, 0.5):
        runs.append(keep(g["sim"](N_granule=1000, N_basket=12, N_perforant=0, p_opto=p, seed=seed, use_tqdm=False), seed))
json.dump(runs, open("sim_runs.json", "w"))
'''
open(f"{SIMDIR}/run/drive.py", "w").write(DRIVER)
t0 = time.time()
out = step("python simulation.py (with the seed check)", [f"{E}/bin/python", "drive.py"], cwd=f"{SIMDIR}/run", timeout=1800)
print("ran in %.1f min" % ((time.time() - t0) / 60))
print("the script's output:\n" + "\n".join(l for l in out.stdout.splitlines() if l.startswith("With p_opto")))
SIM = json.load(open(f"{SIMDIR}/run/sim_runs.json"))


In [ ]:
# The claims file's readings. An event is a run of 1 ms bins whose spike count (all 1012 cells) is above the
# script's closed-loop threshold x_opto (opto_threshold, 100 spikes in a 1 ms step): (a) their total duration,
# (b) their number, (c) the spikes they hold. The criterion is read at the script's seed, 20.
rows = []
for r in SIM:
    c = np.asarray(r["counts"])
    above = c > r["threshold"]
    edges = np.diff(np.r_[0, above.astype(int), 0])
    rows.append({"seed": r["seed"], "p_opto": r["p_opto"], "cells": r["n_cells"], "ms": len(c),
                 "spikes": int(c.sum()), "max_per_ms": int(c.max()), "opto_events": len(r["opto_times"]),
                 "a_duration_ms": int(above.sum()), "b_events": int((edges == 1).sum()), "c_spikes_in_events": int(c[above].sum())})
sim = pd.DataFrame(rows)
print(sim.to_string(index=False))
main = sim[sim.seed == 20].set_index("p_opto")
for m in ("a_duration_ms", "b_events", "c_spikes_in_events"):
    v = main[m]
    low = bool((v.loc[[0.1, 0.25]] < v.loc[0.0]).any())
    print(f"seed 20, {m}: {v.tolist()} at p_opto 0, 0.10, 0.25, 0.50; lower at 0.25 or less than at 0: {low}; "
          f"lower at 0.50 than at 0: {bool(v.loc[0.5] < v.loc[0.0])}; criterion met: {low and bool(v.loc[0.5] < v.loc[0.0])}")
print("\nper seed, value at 0.10, 0.25, 0.50 as a fraction of the value at 0:")
for m in ("a_duration_ms", "b_events", "c_spikes_in_events"):
    w = sim.pivot(index="seed", columns="p_opto", values=m)
    print(m); print((w[[0.1, 0.25, 0.5]].div(w[0.0], axis=0)).round(3).to_string())


**Result: reproduced.** The published
environment pins NEST 3.4 from conda-forge and NESTML 5. PyPI has no NEST 3.4 (its releases are
3.9.0.post0.dev14 and 3.10.0), and with NEST 3.10.0 and NESTML 5.3.0, the newest 5.x, the script's
first step, `generate_nest_target`, fails (`nest.ll_api` has no attribute `sli_func`). With NEST 3.4
from conda-forge and NESTML 5.3.0, the model builds after two changes to the environment, neither
to the model: odetoolbox, which the environment file leaves unpinned and which now resolves to a
release NESTML 5.3.0 cannot call (`simplify_expression`), is set to 2.5, and the conda-forge
`nest-config`, which prints no compiler, is made to print the environment's. The script then runs
as published, and gives the four lines it prints (mean rate 80.65 Hz at fraction 0, 60.06, 52.49
and 49.65 Hz at 0.10, 0.25 and 0.50). At its seed, 20, with events as runs of 1 ms bins above the
closed-loop threshold of 100 spikes, the total event duration is 278 ms at fraction 0 and 124, 37
and 4 ms at 0.10, 0.25 and 0.50, and the spikes in events 52,366 against 17,594, 4,086 and 441, so
readings (a) and (c) meet the criterion; the number of events, (b), is 10, 10, 10 and 4, lower
only at 0.50. With seeds 21, 23 and 24, readings (a) and (c) are lower at every fraction than at
0 (at most 0.70 of the value at 0). Seed 22 meets the criterion under no reading: the network has
1 ms above threshold at fraction 0, and at 0.50 the event duration is the same and the spikes in
events 103 at 0.50 against 101 at 0. The verdict uses the script's own seed, fixed before any analysis. The
outcome verified in round 2 was blocked (not_runnable), before the model was run; the round-3
verifier agreed with the change to reproduced.


## Summary

| Claim | Outcome |
|---|---|
| table1-cohort | partially reproduced |
| units-per-recording | blocked: data_missing |
| transduction-range | blocked: data_missing |
| reporter-gcl-pyramidal | blocked: data_missing |
| patch-hckcr1-hyperpolarizing | blocked: data_missing |
| simulation-dampening | reproduced |
| fig3a-spontaneous-reduction | blocked: data_missing |
| physio-bic-reduction-all-slices | blocked: data_missing |
| physio-bic-amplitude-unchanged | blocked: data_missing |
| table2-fr-decrease-physio-bic | blocked: data_missing |
| table2-fr-decrease-0mg | blocked: data_missing |
| zero-mg-reduction | blocked: data_missing |
| zero-mg-isolated-bursts | blocked: data_missing |
| ka-rhythmic-bursts | blocked: data_missing |
| ka-light-reduction | blocked: data_missing |
| ka-bursts-during-light | blocked: data_missing |
| ka-theta-coherence | blocked: data_missing |
| fig4-firing-propagation | blocked: data_missing |
| fig4-theta-propagation | blocked: data_missing |
| closed-loop-bicuculline | blocked: data_missing |
| intensity-threshold | blocked: data_missing |
| chr2-increase | blocked: data_missing |
| transduction-not-explaining-response | blocked: data_missing |
| s3-reduction-by-condition | blocked: data_missing |
| waveform-clusters | blocked: data_missing |
| waveform-gcl-association | blocked: data_missing |
| opto-response-clusters | blocked: data_missing |
| opto-waveform-association | blocked: data_missing |
| opto-cluster3-profile | blocked: data_missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Only the cohort claim is tested on the dandiset,
and it is tested on the file names and subject metadata. The network model is run as published,
with odetoolbox set to 2.5 and the compiler given to `nest-config`, and reproduces the claim at
the script's seed. The variant on 3C uses threshold crossings on 64 channels
in place of curated units, and is confounded by amplifier saturation at light onset, with or
without the first second of each window.

**What was found about the dataset.** The 69 files hold raw HD-MEA voltage only: no sorted units,
no light events (the GPIO bit and the stimulation log the Methods describe), no condition, medium
or opsin labels, no session description, and no link from a file to the paper's recording names.
Light periods can be recovered from a voltage step in the 3C recordings (every channel group),
where light onset drives most channels to the ends of their range, and in 4C-1, 4C-3, 7D-0,
7D-2, 7D-5 and 7D-7 (part of the array only, steps of 40 to 88 counts); no file of 5C, 8D, 10F
or 11G shows one. In at least one 5C recording, 34% of the first channel
group sits at the ends of the range in the median 100 ms bin. `subject_id` is the
recording, so the archive counts 69 subjects for 7 patients, and every session start time is the
conversion date. No recording is deposited twice.

In [ ]:
print("notebook ran in %.1f min; peak memory %.2f GB" % ((time.time() - T0) / 60,
      resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))